In [1]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

INPUT_CSV = "reviewer_artifact_obshet_FIXED1.csv"
OUTPUT_SUMMARY = "hierarchical_bootstrap_summary.csv"
OUTPUT_CONTRASTS = "hierarchical_bootstrap_ictal_contrasts.csv"

N_BOOT = 2000
SEED = 20260816

STAGES = ["ictal", "interictal", "postictal", "preictal"]

METRICS = [
    "gamma_est",
    "phi_est",
    "r_est",
    "delta_nll_obshet",
    "delta_bic_obshet",
    "delta_nll_gamma0",
    "delta_bic_gamma0",
]

KEY_COLS = [
    "patient",
    "edf_file",
    "seizure_id",
    "channel",
    "band",
    "block",
]

# ------------------------------------------------------------
# Read and clean results
# ------------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

if "success" in df.columns:
    df = df[df["success"].eq(True)].copy()

df = df.drop_duplicates(subset=KEY_COLS, keep="last").copy()

required = {
    "patient",
    "edf_file",
    "seizure_id",
    "period",
    *METRICS,
}

missing = required.difference(df.columns)

if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# A seizure/event is uniquely identified within patient and EDF file.
df["event_id"] = (
    df["patient"].astype(str)
    + "|"
    + df["edf_file"].astype(str)
    + "|"
    + df["seizure_id"].astype(str)
)

print("Rows used:", len(df))
print("\nPatients:", df["patient"].nunique())
print("Seizure events:", df["event_id"].nunique())
print("\nWindows by stage:")
print(df["period"].value_counts())

# ------------------------------------------------------------
# Hierarchical resampling
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)
patients = df["patient"].unique()

def hierarchical_sample(data, rng):
    """Resample patients, then seizures within each selected patient."""

    sampled_patients = rng.choice(
        patients,
        size=len(patients),
        replace=True,
    )

    sampled_parts = []

    for patient_draw, patient in enumerate(sampled_patients):

        patient_df = data[data["patient"] == patient]
        events = patient_df["event_id"].unique()

        sampled_events = rng.choice(
            events,
            size=len(events),
            replace=True,
        )

        for event_draw, event in enumerate(sampled_events):
            event_df = patient_df[
                patient_df["event_id"] == event
            ].copy()

            # Artificial IDs preserve repeated bootstrap draws.
            event_df["bootstrap_patient"] = patient_draw
            event_df["bootstrap_event"] = (
                f"{patient_draw}_{event_draw}"
            )

            sampled_parts.append(event_df)

    return pd.concat(sampled_parts, ignore_index=True)

# ------------------------------------------------------------
# Observed estimates
# ------------------------------------------------------------

observed = (
    df.groupby("period")[METRICS]
      .median()
      .reset_index()
)

# Win rates are proportions with positive differences.
for comparison in ["obshet", "gamma0"]:
    observed[f"joint_win_rate_{comparison}_nll"] = (
        df.groupby("period")[f"delta_nll_{comparison}"]
          .apply(lambda x: np.mean(x > 0))
          .reindex(observed["period"])
          .to_numpy()
    )

    observed[f"joint_win_rate_{comparison}_bic"] = (
        df.groupby("period")[f"delta_bic_{comparison}"]
          .apply(lambda x: np.mean(x > 0))
          .reindex(observed["period"])
          .to_numpy()
    )

# ------------------------------------------------------------
# Bootstrap
# ------------------------------------------------------------

bootstrap_rows = []

for bootstrap_id in range(N_BOOT):

    sample = hierarchical_sample(df, rng)

    for stage, stage_df in sample.groupby("period"):

        row = {
            "bootstrap": bootstrap_id,
            "period": stage,
        }

        for metric in METRICS:
            row[metric] = stage_df[metric].median()

        for comparison in ["obshet", "gamma0"]:
            row[f"joint_win_rate_{comparison}_nll"] = np.mean(
                stage_df[f"delta_nll_{comparison}"] > 0
            )
            row[f"joint_win_rate_{comparison}_bic"] = np.mean(
                stage_df[f"delta_bic_{comparison}"] > 0
            )

        bootstrap_rows.append(row)

bootstrap = pd.DataFrame(bootstrap_rows)

# ------------------------------------------------------------
# Confidence intervals and cluster-aware p-values
# ------------------------------------------------------------

summary_rows = []

summary_metrics = [
    *METRICS,
    "joint_win_rate_obshet_nll",
    "joint_win_rate_obshet_bic",
    "joint_win_rate_gamma0_nll",
    "joint_win_rate_gamma0_bic",
]

for stage in STAGES:

    observed_stage = observed[observed["period"] == stage]
    boot_stage = bootstrap[bootstrap["period"] == stage]

    if observed_stage.empty or boot_stage.empty:
        continue

    for metric in summary_metrics:

        estimate = observed_stage[metric].iloc[0]
        values = boot_stage[metric].dropna().to_numpy()

        lower, upper = np.quantile(values, [0.025, 0.975])

        # For delta metrics, test whether the median difference is zero.
        if metric.startswith("delta_"):
            p_lower = (np.sum(values <= 0) + 1) / (len(values) + 1)
            p_upper = (np.sum(values >= 0) + 1) / (len(values) + 1)
            p_value = min(1.0, 2 * min(p_lower, p_upper))
        else:
            p_value = np.nan

        summary_rows.append({
            "period": stage,
            "metric": metric,
            "estimate": estimate,
            "ci_2.5": lower,
            "ci_97.5": upper,
            "bootstrap_p": p_value,
            "n_windows": int((df["period"] == stage).sum()),
            "n_patients": int(
                df.loc[df["period"] == stage, "patient"].nunique()
            ),
            "n_seizures": int(
                df.loc[df["period"] == stage, "event_id"].nunique()
            ),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_SUMMARY, index=False)

# ------------------------------------------------------------
# Ictal versus nonictal stage contrasts
# ------------------------------------------------------------

contrast_rows = []

for metric in ["gamma_est", "phi_est", "r_est"]:

    observed_medians = df.groupby("period")[metric].median()

    for comparison_stage in [
        "interictal",
        "postictal",
        "preictal",
    ]:

        observed_difference = (
            observed_medians["ictal"]
            - observed_medians[comparison_stage]
        )

        wide = bootstrap.pivot(
            index="bootstrap",
            columns="period",
            values=metric,
        ).dropna(subset=["ictal", comparison_stage])

        differences = (
            wide["ictal"] - wide[comparison_stage]
        ).to_numpy()

        lower, upper = np.quantile(
            differences,
            [0.025, 0.975],
        )

        p_lower = (
            np.sum(differences <= 0) + 1
        ) / (len(differences) + 1)

        p_upper = (
            np.sum(differences >= 0) + 1
        ) / (len(differences) + 1)

        p_value = min(
            1.0,
            2 * min(p_lower, p_upper),
        )

        contrast_rows.append({
            "metric": metric,
            "contrast": f"ictal - {comparison_stage}",
            "estimate": observed_difference,
            "ci_2.5": lower,
            "ci_97.5": upper,
            "bootstrap_p": p_value,
        })

contrasts = pd.DataFrame(contrast_rows)
contrasts.to_csv(OUTPUT_CONTRASTS, index=False)

print("\nHierarchical-bootstrap model comparisons:")
print(
    summary[
        summary["metric"].str.startswith("delta_")
    ].round(4).to_string(index=False)
)

print("\nIctal stage contrasts:")
print(contrasts.round(4).to_string(index=False))

print("\nSaved:")
print(OUTPUT_SUMMARY)
print(OUTPUT_CONTRASTS)

Rows used: 13208

Patients: 14
Seizure events: 105

Windows by stage:
period
interictal    10200
preictal       1548
postictal      1153
ictal           307
Name: count, dtype: int64

Hierarchical-bootstrap model comparisons:
    period           metric  estimate  ci_2.5  ci_97.5  bootstrap_p  n_windows  n_patients  n_seizures
     ictal delta_nll_obshet    0.5714 -0.0397   1.0834       0.0540        307          14          89
     ictal delta_bic_obshet    1.1428 -0.0793   2.1668       0.0540        307          14          89
     ictal delta_nll_gamma0    3.9194  2.7675   5.3333       0.0010        307          14          89
     ictal delta_bic_gamma0    3.7444  1.4406   6.5723       0.0010        307          14          89
interictal delta_nll_obshet    0.3609  0.2746   0.4649       0.0010      10200          14         104
interictal delta_bic_obshet    0.7218  0.5492   0.9297       0.0010      10200          14         104
interictal delta_nll_gamma0    3.0073  1.7784   3.835